# 04 — Ensemble

Combinação dos 3 modelos treinados (ResNet50 + EfficientNetB4 + ViT) via média ponderada de probabilidades.

**Pré-requisito:** notebooks 01, 02 e 03 devem ter sido executados e os checkpoints salvos.

## 1. Setup

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q wandb huggingface_hub timm transformers
    !git clone https://github.com/SEU_USUARIO/tcc-ze-praga-model-playground.git
    %cd tcc-ze-praga-model-playground
    from google.colab import userdata
    import os
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import wandb
import yaml

sys.path.insert(0, str(Path('.').resolve()))
from src.dataset import create_dataloaders, CLASSES
from src.models.resnet50 import build_resnet50
from src.models.efficientnet import build_efficientnet
from src.models.vit import build_vit, ViTWrapper
from src.models.ensemble import EnsembleModel
from src.evaluate import evaluate_model, plot_confusion_matrix

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')

## 2. Configuração

In [ ]:
with open('configs/training_config.yaml') as f:
    base_cfg = yaml.safe_load(f)

DATA_DIR     = '/content/drive/MyDrive/ze-praga-dataset'
CKPT_DIR     = Path('checkpoints')
NUM_CLASSES  = base_cfg['dataset']['num_classes']
IMAGE_SIZE   = base_cfg['dataset']['image_size']
BATCH_SIZE   = base_cfg['training']['batch_size']

## 3. Carregar modelos individuais

In [ ]:
def load_model(build_fn, ckpt_path, device, **kwargs):
    model = build_fn(**kwargs)
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model.to(device)

resnet = load_model(
    build_resnet50, CKPT_DIR / 'best_resnet50.pth', DEVICE,
    num_classes=NUM_CLASSES, pretrained=False
)
effnet = load_model(
    build_efficientnet, CKPT_DIR / 'best_efficientnet.pth', DEVICE,
    num_classes=NUM_CLASSES, pretrained=False
)
vit_base = build_vit(num_classes=NUM_CLASSES, pretrained=False)
vit_base.load_state_dict(torch.load(CKPT_DIR / 'best_vit.pth', map_location=DEVICE))
vit = ViTWrapper(vit_base).eval().to(DEVICE)

print('Modelos carregados.')

## 4. Dataloaders

In [ ]:
_, val_loader, test_loader = create_dataloaders(
    data_dir=DATA_DIR, batch_size=BATCH_SIZE, image_size=IMAGE_SIZE
)

## 5. Avaliação individual (val set)

In [ ]:
individual_results = {}
for name, model in [('ResNet50', resnet), ('EfficientNetB4', effnet), ('ViT', vit)]:
    print(f'\n--- {name} ---')
    r = evaluate_model(model, val_loader, device=DEVICE)
    individual_results[name] = r['accuracy']
    print(f'{name} val_acc: {r["accuracy"]:.4f}')

## 6. Ensemble — média simples

In [ ]:
ensemble = EnsembleModel(
    models=[resnet, effnet, vit],
    weights=None,  # média simples (pesos iguais)
)

print('\n--- Ensemble (média simples) ---')
ens_results = evaluate_model(ensemble, val_loader, device=DEVICE)
individual_results['Ensemble'] = ens_results['accuracy']

fig = plot_confusion_matrix(
    ens_results['y_true'], ens_results['y_pred'],
    save_path='checkpoints/ensemble_val_confusion_matrix.png',
)
plt.show()

## 7. Comparação

In [ ]:
print('\n===== Comparação de Acurácia (val set) =====')
for name, acc in individual_results.items():
    marker = ' ← melhor' if acc == max(individual_results.values()) else ''
    print(f'  {name:<18}: {acc:.4f}{marker}')

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
names = list(individual_results.keys())
accs  = list(individual_results.values())
colors_bar = ['#2D6A4F'] * len(names)
colors_bar[-1] = '#1B4332'  # destaca ensemble
bars = ax.bar(names, accs, color=colors_bar)
ax.set_ylim(max(0, min(accs) - 0.05), 1.0)
ax.set_ylabel('Acurácia')
ax.set_title('Comparação de Modelos — Val Set')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.4f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('checkpoints/model_comparison.png', dpi=150)
plt.show()

## 8. Salvar ensemble

In [ ]:
# Salva os pesos do ensemble (referência aos pesos individuais)
torch.save({
    'resnet50_weights':     torch.load(CKPT_DIR / 'best_resnet50.pth'),
    'efficientnet_weights': torch.load(CKPT_DIR / 'best_efficientnet.pth'),
    'vit_weights':          torch.load(CKPT_DIR / 'best_vit.pth'),
    'ensemble_weights':     [1/3, 1/3, 1/3],
}, CKPT_DIR / 'best_ensemble.pth')

print('Ensemble salvo em checkpoints/best_ensemble.pth')